# Versão modular · mesma referência científica
As funções estão em src/hai_tcc; o notebook apresenta cada passo e seus resultados.
Configuração: configs/baseline.yaml. Saídas: runs/baseline_modular/.
Cada etapa começa em um kernel novo e lê os artefatos salvos em disco.


In [1]:
from pathlib import Path
import sys
for BASE_DIR in (Path.cwd(), *Path.cwd().parents):
    if (BASE_DIR / "configs/baseline.yaml").is_file() and (BASE_DIR / "src/hai_tcc").is_dir():
        break
else:
    raise FileNotFoundError("Abra a pasta do projeto.")
if str(BASE_DIR / "src") not in sys.path:
    sys.path.insert(0, str(BASE_DIR / "src"))
from hai_tcc.experiment import abrir_experimento
exp = abrir_experimento(BASE_DIR)
import numpy as np
import pandas as pd

import joblib
from hai_tcc.features import separar_entradas
from hai_tcc.models import prever_nativo, pontuar
from hai_tcc.evaluation import resumir_previsoes, selecionar_limiar, aplicar_limiar, montar_resultados
exp.exigir("treinamento")
print("Validação:", exp.config["particoes"]["validacao"])


Validação: ['test4.csv.gz']


In [2]:
pacotes = exp.carregar_pacotes()
colunas_modelo = pacotes["RF"]["colunas_modelo"]
for nome, pacote in pacotes.items():
    print(nome, "· entradas do estimador:", pacote["modelo"].n_features_in_)


RF · entradas do estimador: 60
IF sem PCA · entradas do estimador: 60
IF com PCA · entradas do estimador: 21


In [3]:
dados_validacao = exp.ler("validacao", colunas_modelo)
X_validacao, y_validacao, meta_validacao = separar_entradas(dados_validacao, colunas_modelo)
print("Registros de validação:", len(X_validacao))
print(y_validacao.value_counts().sort_index())


Registros de validação: 39601
attack
0    38444
1     1157
Name: count, dtype: int64


In [4]:
previsoes_iniciais = {nome: prever_nativo(p, X_validacao) for nome, p in pacotes.items()}
print("Decisões nativas dos três modelos calculadas.")


Decisões nativas dos três modelos calculadas.


In [5]:
resumo_validacao_inicial = resumir_previsoes(y_validacao, previsoes_iniciais)
display(resumo_validacao_inicial.round(4))


,RF,IF sem PCA,IF com PCA
Acurácia,0.9772,0.5985,0.9485
Precisão,0.8832,0.0579,0.3099
Recall,0.2550,0.8349,0.6206
F1-score,0.3957,0.1083,0.4134
Verdadeiros positivos,295.0000,966.0000,718.0000
Verdadeiros negativos,38405.0000,22734.0000,36845.0000
Falsos positivos,39.0000,15710.0000,1599.0000
Falsos negativos,862.0000,191.0000,439.0000
Taxa de falsos positivos,0.0010,0.4086,0.0416
Taxa de falsos negativos,0.7450,0.1651,0.3794


In [6]:
pontuacoes_validacao = {nome: pontuar(p, X_validacao) for nome, p in pacotes.items()}
print("Pontuações calculadas. Maior pontuação significa maior suspeita.")


Pontuações calculadas. Maior pontuação significa maior suspeita.


In [7]:
print("Regra: maior F1 em test4; em empate exato, maior limiar.")
print("Função selecionar_limiar: src/hai_tcc/evaluation.py")


Regra: maior F1 em test4; em empate exato, maior limiar.
Função selecionar_limiar: src/hai_tcc/evaluation.py


In [8]:
limiares_escolhidos = {nome: selecionar_limiar(y_validacao, s) for nome, s in pontuacoes_validacao.items()}
previsoes_ajustadas = {nome: aplicar_limiar(s, limiares_escolhidos[nome]) for nome, s in pontuacoes_validacao.items()}
display(pd.Series(limiares_escolhidos, name="Limiar").to_frame())
resumo_validacao_ajustada = resumir_previsoes(y_validacao, previsoes_ajustadas)
display(resumo_validacao_ajustada.round(4))


,Limiar
RF,0.170000
IF sem PCA,0.548756
IF com PCA,0.567999


,RF,IF sem PCA,IF com PCA
Acurácia,0.9799,0.9405,0.9826
Precisão,0.6708,0.2125,0.7995
Recall,0.6145,0.3829,0.5376
F1-score,0.6414,0.2733,0.6429
Verdadeiros positivos,711.0000,443.0000,622.0000
Verdadeiros negativos,38095.0000,36802.0000,38288.0000
Falsos positivos,349.0000,1642.0000,156.0000
Falsos negativos,446.0000,714.0000,535.0000
Taxa de falsos positivos,0.0091,0.0427,0.0041
Taxa de falsos negativos,0.3855,0.6171,0.4624


In [9]:
pacote_validacao = {
    "limiares": limiares_escolhidos, "arquivo_validacao": exp.config["particoes"]["validacao"][0],
    "criterio": "maior F1-score; em empate, maior limiar", "regra_decisao": "pontuacao >= limiar",
    "pontuacao_rf": "predict_proba da classe 1", "pontuacao_if": "-score_samples",
}
joblib.dump(pacote_validacao, exp.modelos / "limiares_validacao.joblib")
resumo_validacao_inicial.to_csv(exp.resultados / "validacao_inicial.csv", index_label="Metrica")
resumo_validacao_ajustada.to_csv(exp.resultados / "validacao_ajustada.csv", index_label="Metrica")
montar_resultados(meta_validacao, y_validacao, pontuacoes_validacao, previsoes_ajustadas, limiares_escolhidos).to_csv(
    exp.resultados / "validacao_previsoes.csv.gz", index=False, compression="gzip",
)
exp.registrar("validacao", registros=len(y_validacao), limiares=limiares_escolhidos)
print("Limiar e métricas salvos neste experimento.")


Limiar e métricas salvos neste experimento.
